In [3]:
"""
miRNA Model Training Script — v5
==================================
Changes from v4:
  - Added seed family variance features to capture within-family inconsistency:
      · seed_family_pct_up  : % of times this family is upregulated in training data
      · seed_family_entropy : how mixed the family is (0=always same, 1=50/50 split)
      · seed_family_n       : how many observations back this family's stats
  - These replace the blind assumption that all miRNAs in a family behave the same
  - 17/40 families are mixed (both up and down), so this is biologically important
  - seed_family removed from CAT_COLS (TargetEncoder) — variance features carry
    all family signal without redundancy, avoiding the pct_up vs TargetEncoder conflict
  - Permutation importance now computed leak-free (on held-out CV folds)

HOW TO RUN:
  1. Put this file and microrna_clean.xlsx in the same folder
  2. pip install lightgbm scikit-learn category_encoders openpyxl shap
  3. python train_model.py
  → Saves: lgbm_mirna_model.pkl
"""

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import category_encoders as ce
import shap

# Fix global random state for full reproducibility
np.random.seed(42)


# ─────────────────────────────────────────────────────────────
# STEP 1 — Load and prepare data
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Loading and preparing data")
print("=" * 60)

df = pd.read_excel(r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\microrna_clean.xlsx')

df['parasite'] = df['parasite'].str.replace(r'\s+', '', regex=True)
df = df.drop(columns=['microrna', 'infection', 'microrna_group_simplified'])

# Non-conserved / still missing → NaN
df['seed_family'] = df['seed_family'].replace('not_broadly_conserved', np.nan)

# is_conserved flag: 1 if seed family is known, 0 if not
df['is_conserved'] = df['seed_family'].notna().astype(int)

# Interaction feature: parasite × cell type
df['parasite_celltype'] = df['parasite'].astype(str) + '_' + df['cell type'].astype(str)

print(f"Total rows            : {len(df)}")
print(f"Seed family known     : {df['is_conserved'].sum()}")
print(f"Seed family unknown   : {(df['is_conserved']==0).sum()}")
print(f"Target balance        :\n{df['is_upregulated'].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────
# STEP 2 — Seed family variance features
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Computing seed family variance features")
print("=" * 60)

def add_family_variance_features(df_train, df_target):
    """
    Computes per-family stats from df_train and applies them to df_target.
    Always fit on training data only — never on the validation fold.

    Features added:
      seed_family_pct_up  : proportion upregulated in this family (0.0 – 1.0)
                            → 0.5 means perfectly mixed, 0 or 1 means always same direction
      seed_family_entropy : Shannon entropy of up/down split, normalised to 0–1
                            → 0 = always same direction, 1 = perfectly mixed (50/50)
      seed_family_n       : number of observations for this family in training data
                            → low n means stats are unreliable
    """
    family_stats = (
        df_train.dropna(subset=['seed_family'])
        .groupby('seed_family')['is_upregulated']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'pct_up', 'count': 'n'})
    )

    def entropy(p):
        # Binary Shannon entropy, normalised to [0, 1]
        if p <= 0 or p >= 1:
            return 0.0
        return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))  # max is 1.0 at p=0.5

    family_stats['entropy'] = family_stats['pct_up'].apply(entropy)

    # Map stats onto target dataframe
    df_target = df_target.copy()
    df_target['seed_family_pct_up']  = df_target['seed_family'].map(family_stats['pct_up'])
    df_target['seed_family_entropy'] = df_target['seed_family'].map(family_stats['entropy'])
    df_target['seed_family_n']       = df_target['seed_family'].map(family_stats['n'])

    # For rows with unknown seed_family: fill with neutral/missing signal
    # pct_up  → 0.5  (we have no directional information)
    # entropy → 1.0  (maximum uncertainty — we know nothing)
    # n       → 0    (zero observations to back this up)
    df_target['seed_family_pct_up']  = df_target['seed_family_pct_up'].fillna(0.5)
    df_target['seed_family_entropy'] = df_target['seed_family_entropy'].fillna(1.0)
    df_target['seed_family_n']       = df_target['seed_family_n'].fillna(0)

    return df_target

# Apply to the full dataset (used for final model training in Step 5)
df = add_family_variance_features(df, df)

# Show what we computed
print(f"\n{'seed_family':<45} {'n':>4} {'pct_up':>7} {'entropy':>8} {'mixed?':>7}")
print("-" * 70)
sample = df.dropna(subset=['seed_family']).drop_duplicates('seed_family') \
           .sort_values('seed_family_n', ascending=False)
for _, row in sample.iterrows():
    mixed = "YES" if 0 < row['seed_family_pct_up'] < 1 else "no"
    print(f"  {str(row['seed_family']):<43} {int(row['seed_family_n']):>4} "
          f"{row['seed_family_pct_up']:>7.3f} {row['seed_family_entropy']:>8.3f} {mixed:>7}")


# ─────────────────────────────────────────────────────────────
# STEP 3 — Features and target
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — Features and target")
print("=" * 60)

TARGET   = 'is_upregulated'
CAT_COLS = ['parasite', 'organism', 'cell type', 'parasite_celltype']  # seed_family removed — variance features carry all family signal
NUM_COLS = ['time', 'is_conserved',
            'seed_family_pct_up', 'seed_family_entropy', 'seed_family_n']

X = df[CAT_COLS + NUM_COLS]
y = df[TARGET]

print(f"Categorical   : {CAT_COLS}")
print(f"Numeric       : {NUM_COLS}")
print(f"Total features: {len(CAT_COLS) + len(NUM_COLS)}")


# ─────────────────────────────────────────────────────────────
# STEP 4 — Cross-Validation with Optuna best parameters
#          Variance features recomputed inside each fold
#          to prevent any leakage
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Cross-Validation (5-fold, Optuna best params)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BEST_PARAMS = {
    'n_estimators':      303,
    'max_depth':         7,
    'learning_rate':     0.00604,
    'num_leaves':        24,
    'min_child_samples': 6,
    'subsample':         0.9999,
    'colsample_bytree':  0.7345,
    'reg_alpha':         0.01531,
    'reg_lambda':        5.21e-6,
    'random_state':      42,
    'verbose':           -1,
    'class_weight':      'balanced',
}

def make_pipe():
    return Pipeline([
        ('encoder', ce.TargetEncoder(cols=CAT_COLS, smoothing=1.0, handle_missing='value')),
        ('model',   LGBMClassifier(**BEST_PARAMS))
    ])

# Manual CV loop so we can recompute variance features per fold
auc_scores, acc_scores, f1_scores = [], [], []
fold_importances = []

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(df, y)):
    df_train_fold = df.iloc[train_idx].copy()
    df_val_fold   = df.iloc[val_idx].copy()

    # Recompute variance features using ONLY the training fold
    df_train_fold = add_family_variance_features(df_train_fold, df_train_fold)
    df_val_fold   = add_family_variance_features(df_train_fold, df_val_fold)

    X_train = df_train_fold[CAT_COLS + NUM_COLS]
    y_train = df_train_fold[TARGET]
    X_val   = df_val_fold[CAT_COLS + NUM_COLS]
    y_val   = df_val_fold[TARGET]

    fold_pipe = make_pipe()
    fold_pipe.fit(X_train, y_train)

    y_prob = fold_pipe.predict_proba(X_val)[:, 1]
    y_pred = fold_pipe.predict(X_val)

    auc_scores.append(roc_auc_score(y_val, y_prob))
    acc_scores.append(accuracy_score(y_val, y_pred))
    f1_scores.append(f1_score(y_val, y_pred))

    # Leak-free permutation importance on held-out fold
    X_val_enc = fold_pipe.named_steps['encoder'].transform(X_val)
    perm = permutation_importance(
        fold_pipe.named_steps['model'],
        X_val_enc, y_val,
        n_repeats=10, random_state=42, scoring='roc_auc'
    )
    fold_importances.append(perm.importances_mean)
    print(f"  Fold {fold_idx + 1} — AUC: {auc_scores[-1]:.3f}")

auc = np.array(auc_scores)
acc = np.array(acc_scores)
f1  = np.array(f1_scores)

print(f"\nROC-AUC : {auc.mean():.3f} ± {auc.std():.3f}")
print(f"Accuracy: {acc.mean():.3f} ± {acc.std():.3f}")
print(f"F1      : {f1.mean():.3f} ± {f1.std():.3f}")
print(f"AUC per fold: {[round(x, 3) for x in auc.tolist()]}")


# ─────────────────────────────────────────────────────────────
# STEP 5 — Train final model on all data
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Training final model on all data")
print("=" * 60)

pipe = make_pipe()
pipe.fit(X, y)
print("Done.")


# ─────────────────────────────────────────────────────────────
# STEP 6 — Permutation Feature Importance (leak-free, averaged)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Permutation Feature Importance (leak-free)")
print("=" * 60)

mean_importances = np.mean(fold_importances, axis=0)
std_importances  = np.std(fold_importances,  axis=0)

perm_df = pd.DataFrame({
    'feature':    X.columns.tolist(),
    'importance': mean_importances,
    'std':        std_importances
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>10} {'Std':>8}")
print("-" * 55)
for _, row in perm_df.iterrows():
    bar  = '█' * max(0, int(row['importance'] * 60))
    sign = '+' if row['importance'] >= 0 else ''
    print(f"  {row['feature']:<23} {sign}{row['importance']:.4f} ± {row['std']:.4f}  {bar}")


# ─────────────────────────────────────────────────────────────
# STEP 7 — Build SHAP explainer
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Building SHAP explainer")
print("=" * 60)

X_enc = pipe.named_steps['encoder'].transform(X)
explainer = shap.TreeExplainer(pipe.named_steps['model'])
shap_vals = explainer.shap_values(X_enc)
print(f"SHAP values shape : {np.array(shap_vals).shape}  ✓")


# ─────────────────────────────────────────────────────────────
# STEP 8 — Save model bundle
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8 — Saving model bundle")
print("=" * 60)

model_bundle = {
    'model':     pipe,
    'encoder':   pipe.named_steps['encoder'],
    'lgbm':      pipe.named_steps['model'],
    'explainer': explainer,
    'metrics': {
        'auc_mean':   round(auc.mean(), 3),
        'auc_std':    round(auc.std(),  3),
        'acc_mean':   round(acc.mean(), 3),
        'acc_std':    round(acc.std(),  3),
        'f1_mean':    round(f1.mean(),  3),
        'f1_std':     round(f1.std(),   3),
        'auc_folds':  [round(x, 3) for x in auc.tolist()],
        'best_params': BEST_PARAMS,
        'n_train':    len(X),
        'feature_importance': perm_df.to_dict('records'),
    },
    'options': {
        'parasite':      sorted(df['parasite'].dropna().unique().tolist()),
        'organism':      sorted(df['organism'].dropna().unique().tolist()),
        'cell_type':     sorted(df['cell type'].dropna().unique().tolist()),
        'time':          sorted(df['time'].dropna().unique().tolist()),
        'seed_families': sorted(df['seed_family'].dropna().unique().tolist()),
    },
    'cat_cols':      CAT_COLS,
    'feature_names': list(X.columns),
    # Store family stats computed on full data — needed at prediction time
    'family_stats':  df.dropna(subset=['seed_family'])
                       .drop_duplicates('seed_family')
                       .set_index('seed_family')[['seed_family_pct_up',
                                                   'seed_family_entropy',
                                                   'seed_family_n']]
                       .to_dict('index'),
}

with open(r'C:\Users\MSI\Desktop\PFE\PHASE 2\seed_family_code\lgbm_mirna_model_fixed.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print("Saved: lgbm_mirna_model.pkl")

STEP 1 — Loading and preparing data
Total rows            : 206
Seed family known     : 138
Seed family unknown   : 68
Target balance        :
is_upregulated
1    103
0    103

STEP 2 — Computing seed family variance features

seed_family                                      n  pct_up  entropy  mixed?
----------------------------------------------------------------------
  let-7-5p/miR-98-5p                            16   0.812    0.696     YES
  miR-30-5p/384-5p                               7   0.714    0.863     YES
  miR-29-3p                                      7   0.714    0.863     YES
  miR-15-5p/16-5p/195-5p/424-5p/497-5p           7   0.000    0.000      no
  miR-191-5p                                     6   0.167    0.650     YES
  miR-291-3p/294-3p/295-3p/302-3p                6   0.500    1.000     YES
  miR-9-5p                                       6   0.833    0.650     YES
  miR-26-5p                                      6   0.333    0.918     YES
  miR-23-3p       